In [0]:
storage_account_name = "vasudatalake1"
container_name = "landing"
access_key = "<storageKey>"
mount_point = "/mnt/landing"
 
configs = {
  f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key
}
 
try:
    if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
        print(f"{mount_point} is already mounted.")
    else:
        dbutils.fs.mount(
          source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/",
          mount_point = mount_point,
          extra_configs = configs
        )
        print("✅ Successfully mounted ADLS Gen2!")
except Exception as e:
    print(f"❌ Mount failed. Error: {str(e)[:200]}")
 

In [0]:
print("--- 1. Testing Write Access ---")
# Create some quick dummy data
dummy_data = [("1", "Task 1 Complete", "Success")]
df_test = spark.createDataFrame(dummy_data, ["ID", "Task", "Status"])

# Write the data into your landing container
write_path = "/mnt/landing/validation_test.csv"
df_test.write.mode("overwrite").csv(write_path, header=True)
print("✅ Write Test Passed! File saved to ADLS.")


print("\n--- 2. Testing Read Access ---")
# Read the file back from the landing container
df_read = spark.read.csv(write_path, header=True)
display(df_read)
print("✅ Read Test Passed!")

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("/mnt/landing/reference/olist_customers_dataset.csv")
 
display(df)
 